# Budget Management - Time Series Forecasting Model

Upload `expenses_feature_engineered_full.csv`, train the model, and export the final `.pkl` file.

In [ ]:
# Step 1: Upload dataset
from google.colab import files
uploaded = files.upload()

# After upload, the CSV should appear in the Colab file list.


In [ ]:
# Step 2: Import libraries
import pandas as pd
import numpy as np
import pickle
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error


In [ ]:
# Step 3: Load dataset
CSV_PATH = "expenses_feature_engineered_full.csv"

df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


In [ ]:
# Step 4: Prepare features and target

TARGET_COL = "total_amount"
CATEGORY_COL = "refined_category"

feature_cols = [c for c in df.columns if c not in [TARGET_COL, CATEGORY_COL]]

df_sorted = df.sort_values(["year", "month", "refined_category"]).reset_index(drop=True)

X = df_sorted[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df_sorted[TARGET_COL].astype(float)

print("Number of features:", len(feature_cols))
print("Target:", TARGET_COL)


In [ ]:
# Step 5: Time-based train/test split

split = int(len(df_sorted) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


In [ ]:
# Step 6: Train Random Forest time-series model

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

metrics = {
    "mae": round(float(mae), 2),
    "rmse": round(float(rmse), 2),
    "r2": round(float(r2), 4),
    "training_rows": int(len(X_train)),
    "testing_rows": int(len(X_test)),
    "total_rows": int(len(df_sorted)),
    "date_range": f"{int(df['year'].min())}-{int(df['month'].min()):02d} to {int(df['year'].max())}-{int(df['month'].max()):02d}",
}

metrics


In [ ]:
# Step 7: Save final PKL model

categories = sorted(df["refined_category"].dropna().astype(str).unique().tolist())

bundle = {
    "model_name": "RandomForestRegressor time-series expense forecasting model",
    "model": model,
    "feature_columns": feature_cols,
    "target_column": TARGET_COL,
    "category_column": CATEGORY_COL,
    "metrics": metrics,
    "categories": categories,
    "training_note": "Trained on feature-engineered monthly category expense data.",
    "sklearn_version": sklearn.__version__,
}

PKL_NAME = "budget_time_series_model.pkl"

with open(PKL_NAME, "wb") as file:
    pickle.dump(bundle, file)

print("PKL file saved:", PKL_NAME)
print("Metrics:", metrics)


In [ ]:
# Step 8: Download final PKL file

files.download("budget_time_series_model.pkl")


## Viva/report explanation

This model uses historical monthly expense data with lag features, rolling averages, seasonal/time features, and category encoding. It predicts future monthly expenses category-wise. The predicted total can be compared with the user's monthly budget to identify budget overrun risk.